In [1]:
import os
import random
import re
from pathlib import Path

# ---------------- CONFIG ----------------
SRC_DIR = "verilog_benchmark_circuits"
DST_DIR = "verilog_benchmark_circuits_Trojan"
os.makedirs(DST_DIR, exist_ok=True)

SEED = 42
random.seed(SEED)

# ----------------------------------------
# Utility: extract signals
# ----------------------------------------

def extract_signals(verilog_text):
    inputs, outputs, wires = [], [], []

    for line in verilog_text.splitlines():
        line = line.split("//")[0].strip()
        if line.startswith("input"):
            toks = line.replace("input", "").replace(";", "")
            inputs += [x.strip() for x in toks.split(",")]
        elif line.startswith("output"):
            toks = line.replace("output", "").replace(";", "")
            outputs += [x.strip() for x in toks.split(",")]
        elif line.startswith("wire"):
            toks = line.replace("wire", "").replace(";", "")
            wires += [x.strip() for x in toks.split(",")]

    return inputs, outputs, wires


def inject_before_endmodule(verilog_text, trojan_code):
    lines = verilog_text.splitlines(True)
    for i in range(len(lines)-1, -1, -1):
        if "endmodule" in lines[i]:
            return "".join(lines[:i]) + "\n" + trojan_code + "\n" + "".join(lines[i:])
    return verilog_text


# ----------------------------------------
# Gate-level Trojan generators
# ----------------------------------------

def trojan_andxor(tid, inputs, outputs):
    if len(inputs) < 2 or len(outputs) == 0:
        return None

    trig1, trig2 = random.sample(inputs, 2)
    victim = random.choice(outputs)

    return f"""
// -------- Trojan AND+XOR --------
wire troj_trig_{tid};
and U_tand_{tid}(troj_trig_{tid}, {trig1}, {trig2});

wire {victim}_troj_{tid};
xor U_txor_{tid}({victim}_troj_{tid}, {victim}, troj_trig_{tid});
buf U_tbuf_{tid}({victim}, {victim}_troj_{tid});
"""


def trojan_countermux(tid, inputs, outputs):
    if len(inputs) == 0 or len(outputs) == 0:
        return None

    trig = random.choice(inputs)
    victim = random.choice(outputs)

    # 8-bit counter built from XOR chain (structural style)
    counter_wires = "\n".join([
        f"wire troj_cnt_{tid}_{i};"
        for i in range(8)
    ])

    counter_logic = "\n".join([
        f"xor U_cnt_{tid}_{i}(troj_cnt_{tid}_{i}, troj_cnt_{tid}_{i}, {trig});"
        for i in range(8)
    ])

    return f"""
// -------- Trojan Counter+MUX --------
{counter_wires}

{counter_logic}

wire troj_trig_{tid};
and U_tand_{tid}(troj_trig_{tid}, troj_cnt_{tid}_0, troj_cnt_{tid}_1);

wire {victim}_troj_{tid};
and U_tmux1_{tid}({victim}_troj_{tid}, {victim}, troj_trig_{tid});
buf U_tbuf_{tid}({victim}, {victim}_troj_{tid});
"""


def trojan_fsmor(tid, inputs, outputs):
    if len(inputs) == 0 or len(outputs) == 0:
        return None

    trig = random.choice(inputs)
    victim = random.choice(outputs)

    return f"""
// -------- Trojan FSM+OR --------
wire troj_state_{tid}_0, troj_state_{tid}_1;
xor U_fsm0_{tid}(troj_state_{tid}_0, {trig}, {trig});
xor U_fsm1_{tid}(troj_state_{tid}_1, troj_state_{tid}_0, {trig});

wire troj_trig_{tid};
and U_tand_{tid}(troj_trig_{tid}, troj_state_{tid}_0, troj_state_{tid}_1);

wire {victim}_troj_{tid};
or U_tor_{tid}({victim}_troj_{tid}, {victim}, troj_trig_{tid});
buf U_tbuf_{tid}({victim}, {victim}_troj_{tid});
"""


TROJAN_FUNCS = {
    "andxor": trojan_andxor,
    "countermux": trojan_countermux,
    "fsmor": trojan_fsmor,
}

# ----------------------------------------
# Apply Trojan to all circuits
# ----------------------------------------

def inject_all():
    files = sorted(Path(SRC_DIR).glob("*.v"), key=lambda p: p.stat().st_size)

    for f in files:
        text = f.read_text()

        inputs, outputs, wires = extract_signals(text)

        for tname, tfunc in TROJAN_FUNCS.items():
            tid = random.randint(1000, 9999)

            troj_code = tfunc(tid, inputs + wires, outputs)
            if troj_code is None:
                continue

            new_text = inject_before_endmodule(text, troj_code)

            outname = f.stem + f"__trojan_{tname}.v"
            outpath = Path(DST_DIR) / outname
            outpath.write_text(new_text)

            print(f"[OK] {f.name} -> {outname}")


if __name__ == "__main__":
    inject_all()
    print("\nTrojanized netlists generated successfully.")

[OK] c17.v -> c17__trojan_andxor.v
[OK] c17.v -> c17__trojan_countermux.v
[OK] c17.v -> c17__trojan_fsmor.v
[OK] c432.v -> c432__trojan_andxor.v
[OK] c432.v -> c432__trojan_countermux.v
[OK] c432.v -> c432__trojan_fsmor.v
[OK] c499.v -> c499__trojan_andxor.v
[OK] c499.v -> c499__trojan_countermux.v
[OK] c499.v -> c499__trojan_fsmor.v
[OK] ctrl.v -> ctrl__trojan_andxor.v
[OK] ctrl.v -> ctrl__trojan_countermux.v
[OK] ctrl.v -> ctrl__trojan_fsmor.v
[OK] c880.v -> c880__trojan_andxor.v
[OK] c880.v -> c880__trojan_countermux.v
[OK] c880.v -> c880__trojan_fsmor.v
[OK] int2float.v -> int2float__trojan_andxor.v
[OK] int2float.v -> int2float__trojan_countermux.v
[OK] int2float.v -> int2float__trojan_fsmor.v
[OK] router.v -> router__trojan_andxor.v
[OK] router.v -> router__trojan_countermux.v
[OK] router.v -> router__trojan_fsmor.v
[OK] c1908.v -> c1908__trojan_andxor.v
[OK] c1908.v -> c1908__trojan_countermux.v
[OK] c1908.v -> c1908__trojan_fsmor.v
[OK] c1355.v -> c1355__trojan_andxor.v
[OK] c1